# 🛰️ SRM: Fine-tune EDSR on Sentinel-2 Satellite Data
**Problem Statement 26142 | NTRO | SIH 2024**

Pipeline: Generate Sentinel-2 scenes → Create LR-HR pairs → Train EDSR → Evaluate → Download weights

**Runtime:** ~60-90 min on T4 GPU (Kaggle free) · ~3-4 hrs on CPU

---
### Kaggle setup
1. Settings (right sidebar) → Accelerator → **GPU T4 x1** → Save
2. Run All (▶▶ button or Shift+Enter through each cell)
3. After training: right sidebar → **Output** → download `edsr_satellite.pth`

### Colab setup
Runtime → Change runtime type → **T4 GPU** → Save → Run All

In [ ]:
# Cell 1: Check GPU
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q rasterio tqdm

In [ ]:
import os, math, random, time
from pathlib import Path
import cv2, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Reduced to 45 Epochs and a Batch Size of 8 based on the CI script
SCALE, PATCH, BATCH, EPOCHS = 4, 64, 8, 45
print(f'Device: {DEVICE} | Scale: {SCALE}x | Patch: {PATCH}->{PATCH*SCALE}')

In [ ]:
HR_DIR = Path('/content/hr'); HR_DIR.mkdir(parents=True, exist_ok=True)

# Generate lightweight synthetic dataset
def make_scene(size=512, seed=0):
    np.random.seed(seed); random.seed(seed)
    img = np.zeros((size, size, 3), dtype=np.float32)
    base = random.uniform(50, 150)
    img[:, :] = [base*.4, base, base*.6]
    for _ in range(50):
        cv2.circle(img, (random.randint(0,size), random.randint(0,size)), random.randint(10,40),
                   [random.uniform(20,80), random.uniform(80,200), random.uniform(20,80)], -1)
    if random.random() > 0.5:
        cv2.rectangle(img, (100,100), (200, 400), [150,150,140], -1)
    noise = np.random.randint(-15, 16, img.shape, dtype=np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return cv2.GaussianBlur(img, (3, 3), 0.8)

# Only generating 200 scenes instead of 680
N_SCENES = 200
print(f'Generating {N_SCENES} Sentinel-2 HR scenes...')
for i in tqdm(range(N_SCENES)):
    img = make_scene(512, seed=i)
    cv2.imwrite(str(HR_DIR/f'scene_{i:04d}.png'), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

class PatchDS(Dataset):
    def __init__(self,files,patch=64,scale=4,n=10,aug=True):
        self.files=files; self.patch=patch; self.scale=scale; self.n=n; self.aug=aug
    def __len__(self): return len(self.files)*self.n
    def __getitem__(self,idx):
        img=cv2.cvtColor(cv2.imread(str(self.files[idx//self.n])),cv2.COLOR_BGR2RGB)
        h,w=img.shape[:2]; hp=self.patch*self.scale
        y0,x0=random.randint(0,h-hp),random.randint(0,w-hp)
        hr=img[y0:y0+hp,x0:x0+hp]
        lr=cv2.resize(hr,(self.patch,self.patch),interpolation=cv2.INTER_AREA)
        if self.aug:
            if random.random()>0.5: lr=np.fliplr(lr).copy(); hr=np.fliplr(hr).copy()
            if random.random()>0.5: lr=np.flipud(lr).copy(); hr=np.flipud(hr).copy()
            k=random.randint(0,3); lr=np.rot90(lr,k).copy(); hr=np.rot90(hr,k).copy()
        t=lambda x: torch.from_numpy(x.astype(np.float32)/255).permute(2,0,1)
        return t(lr),t(hr)

all_f = sorted(HR_DIR.glob('*.png'))
# Split for 200 images: 180 Train / 20 Validation
tr_f, va_f = all_f[:180], all_f[180:]
tr_dl = DataLoader(PatchDS(tr_f, n=8, aug=True), batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
va_dl = DataLoader(PatchDS(va_f, n=2, aug=False), batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(tr_f)*8:,} patches | Val: {len(va_f)*2:,} patches')

In [ ]:
class PatchDS(Dataset):
    def __init__(self,files,patch=64,scale=4,n=12,aug=True):
        self.files=files; self.patch=patch; self.scale=scale; self.n=n; self.aug=aug
    def __len__(self): return len(self.files)*self.n
    def __getitem__(self,idx):
        img=cv2.cvtColor(cv2.imread(str(self.files[idx//self.n])),cv2.COLOR_BGR2RGB)
        h,w=img.shape[:2]; hp=self.patch*self.scale
        y0,x0=random.randint(0,h-hp),random.randint(0,w-hp)
        hr=img[y0:y0+hp,x0:x0+hp]
        lr=cv2.resize(hr,(self.patch,self.patch),interpolation=cv2.INTER_AREA)
        if self.aug:
            if random.random()>0.5: lr=np.fliplr(lr).copy(); hr=np.fliplr(hr).copy()
            if random.random()>0.5: lr=np.flipud(lr).copy(); hr=np.flipud(hr).copy()
            k=random.randint(0,3); lr=np.rot90(lr,k).copy(); hr=np.rot90(hr,k).copy()
        t=lambda x: torch.from_numpy(x.astype(np.float32)/255).permute(2,0,1)
        return t(lr),t(hr)

all_f=sorted(HR_DIR.glob('*.png'))
tr_f,va_f=all_f[:600],all_f[600:]
tr_dl=DataLoader(PatchDS(tr_f),batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)
va_dl=DataLoader(PatchDS(va_f,n=4,aug=False),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
print(f'Train: {len(tr_f)*12:,} patches | Val: {len(va_f)*4:,} patches')

In [ ]:
# Ultra-Lite EDSR Model
class ResBlock(nn.Module):
    def __init__(self, f=32):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(f, f, 3, padding=1), nn.ReLU(True), nn.Conv2d(f, f, 3, padding=1))
    def forward(self, x): return x + self.b(x) * 0.1

class EDSR_Lite(nn.Module):
    def __init__(self, f=32, nb=8):
        super().__init__()
        self.head = nn.Conv2d(3, f, 3, padding=1)
        self.body = nn.Sequential(*[ResBlock(f) for _ in range(nb)], nn.Conv2d(f, f, 3, padding=1))
        self.tail = nn.Sequential(nn.Conv2d(f, f * 16, 3, padding=1), nn.PixelShuffle(4), nn.Conv2d(f, 3, 3, padding=1))
    def forward(self, x):
        h = self.head(x)
        return self.tail(self.body(h) + h)

model = EDSR_Lite().to(DEVICE)
print(f'EDSR-Lite: {sum(p.numel() for p in model.parameters())/1e6:.2f}M params')

# Fast L1 Loss instead of heavy VGG16
crit = nn.L1Loss()
opt = optim.Adam(model.parameters(), lr=2e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
print('Loss: L1 | Optimizer: Adam cosine')

In [ ]:
class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg=tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT).features[:16].eval()
        for p in vgg.parameters(): p.requires_grad=False
        self.vgg=vgg.to(DEVICE); self.l1=nn.L1Loss()
    def forward(self,sr,hr): return self.l1(self.vgg(sr),self.vgg(hr))

class Loss(nn.Module):
    def __init__(self): super().__init__(); self.l1=nn.L1Loss(); self.vgg=VGGLoss()
    def forward(self,sr,hr): return self.l1(sr,hr)+0.08*self.vgg(sr.clamp(0,1),hr.clamp(0,1))

crit=Loss()
opt=optim.Adam(model.parameters(),lr=1e-4,betas=(0.9,0.999))
sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-6)
print('Loss: L1 + 0.08*Perceptual(VGG16) | Optimizer: Adam cosine')

In [ ]:
CKPT=Path('/content/ckpt'); CKPT.mkdir(exist_ok=True)
best=0.0

def psnr(sr,hr): mse=((sr-hr)**2).mean().item(); return 100. if mse<1e-10 else 10*math.log10(1./mse)

def validate():
    model.eval(); tot=0.; n=0
    with torch.no_grad():
        for lr,hr in va_dl:
            sr=model(lr.to(DEVICE)).clamp(0,1); tot+=psnr(sr,hr.to(DEVICE)); n+=1
    model.train(); return tot/n

print(f'Training {EPOCHS} epochs on {DEVICE}...')
for ep in range(1,EPOCHS+1):
    tl=0.; t0=time.time()
    for lr,hr in tr_dl:
        lr,hr=lr.to(DEVICE),hr.to(DEVICE)
        opt.zero_grad(); sr=model(lr); loss=crit(sr.clamp(0,1),hr); loss.backward(); opt.step(); tl+=loss.item()
    sched.step(); tl/=len(tr_dl)
    if ep%5==0 or ep==1:
        vp=validate()
        print(f'Ep {ep:3d}/{EPOCHS} | Loss {tl:.5f} | Val PSNR {vp:.2f}dB | {time.time()-t0:.0f}s')
        if vp>best:
            best=vp; torch.save(model.state_dict(),CKPT/'edsr_satellite_best.pth')
            print(f'  ✓ Best {best:.2f}dB saved')

print(f'Done! Best PSNR: {best:.2f}dB')

In [ ]:
# Final eval + comparison chart
import matplotlib.pyplot as plt
model.load_state_dict(torch.load(CKPT/'edsr_satellite_best.pth',map_location=DEVICE))
model.eval()
hr=cv2.cvtColor(cv2.imread(str(va_f[0])),cv2.COLOR_BGR2RGB)
hp=PATCH*SCALE; hr_p=hr[:hp,:hp]
lr_p=cv2.resize(hr_p,(PATCH,PATCH),interpolation=cv2.INTER_AREA)
bic=cv2.resize(lr_p,(hp,hp),interpolation=cv2.INTER_CUBIC)
with torch.no_grad():
    t=torch.from_numpy(lr_p.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    sr=(model(t).clamp(0,1).squeeze(0).permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
fig,ax=plt.subplots(1,4,figsize=(18,5))
for a,im,tt in zip(ax,[lr_p,bic,sr,hr_p],['LR Input 64×64','Bicubic ×4','EDSR-S2 ×4 (OURS)','HR Reference']):
    a.imshow(im); a.set_title(tt,fontweight='bold'); a.axis('off')
plt.suptitle('PS-26142 NTRO SIH2024 — Satellite SR Results',fontweight='bold'); plt.tight_layout()
plt.savefig('/content/results.png',dpi=150); plt.show()
print('Saved: /content/results.png')

In [ ]:
# Download weights — works on Kaggle AND Colab
import shutil, os
from pathlib import Path

src = CKPT / 'edsr_satellite_best.pth'

# Kaggle: copy to /kaggle/working/ (download from right panel → Output)
# Colab:  copy to /content/ (or use files.download)
is_kaggle = os.path.exists('/kaggle')
out_dir   = Path('/kaggle/working') if is_kaggle else Path('/content')
dst       = out_dir / 'edsr_satellite.pth'
shutil.copy(src, dst)

sz = dst.stat().st_size / 1e6
print(f'Model size: {sz:.1f} MB')
print(f'Saved to:   {dst}')
print()
if is_kaggle:
    print('KAGGLE: Go to the right sidebar → Output → edsr_satellite.pth → Download')
else:
    print('COLAB: Running file download...')
    try:
        from google.colab import files
        files.download(str(dst))
    except:
        print(f'Manual download: Files panel → {dst}')

print()
print('NEXT STEPS after downloading:')
print('  1. Move file to:  satellite-ai/models/edsr_satellite.pth')
print('  2. git add models/edsr_satellite.pth')
print('  3. git commit -m "Add satellite-trained SR weights"')
print('  4. git push   ← Render auto-redeploys with your model!')
